# vLLM Server on Kaggle (T4 GPU)
Notebook này khởi chạy vLLM server và sử dụng `ngrok` (hoặc `localtunnel`) để public endpoint ra ngoài, cho phép `lab28-platform-api` ở local kết nối tới.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt vLLM và pyngrok
!pip install -q "vllm==0.26.0" pyngrok

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 3. Lấy Ngrok Token từ Kaggle Secrets (An toàn, không hardcode)
try:
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
except Exception as e:
    print("Vui lòng tạo secret NGROK_AUTH_TOKEN trong Add-ons -> Secrets.")
    NGROK_AUTH_TOKEN = input("Hoặc nhập trực tiếp token của bạn tại đây: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# 4. Khởi chạy vLLM Server dưới nền
# Sử dụng model Qwen3-4B-Instruct-2507 như yêu cầu, với gpu-memory-utilization=0.85
vllm_process = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen3-4B-Instruct-2507",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85"
])

print("Đang khởi động vLLM (Mất khoảng 2-3 phút để tải model)...")
time.sleep(30)

In [ ]:
# 5. Tạo đường hầm (Tunnel) bằng ngrok
public_url = ngrok.connect(8000).public_url
print("="*60)
print(f"vLLM OpenAI Endpoint URL: {public_url}/v1")
print("\nCopy URL trên và cập nhật vào file .env ở local của bạn:")
print(f"LAB28_VLLM_BASE_URL={public_url}/v1")
print("LAB28_VLLM_MODEL_ID=Qwen/Qwen3-4B-Instruct-2507")
print("LAB28_VLLM_REQUIRE_REAL=true")
print("="*60)